# 03 — Portfolio KPIs, ranked spot book, hypothesis board

Hedge-fund read: Sharpe, Sortino, Calmar, Ulcer, max DD, CAGR, turnover, names held. Ranked allocation is the QMIE allocator idea on **daily spot** (lookback ROC, top-3, cluster_max=1). It does not execute. `quantity` stays 0 in production.

## H7

Ranked top-N eligible names beat equal-weight eligible names on OOS Sharpe or DD.

After the board: if the crypto model is not robust under chronological OOS + DF, **do not ship parameter changes**.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "research").exists():
    pass
elif (ROOT / "python" / "research").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root", ROOT)


In [ ]:
import json
from pathlib import Path
import pandas as pd
from research.trend_lab.allocation import bh_equal, book_kpis, equal_weight_book, ranked_spot_book
from research.trend_lab.data import CORE, load_symbol
from research.trend_lab.plots import equity_overlay
from research.trend_lab.protocol import split_frame
from research.trend_lab.spot_system import SpotParams, spot_signal
from research.trend_lab.protocol import WARMUP_BARS

btc, _ = load_symbol("BTCUSDT", "1d")
parts = split_frame(btc)
close_cols, held_cols = {}, {}
for sym in CORE[:4]:
    df, _ = load_symbol(sym, "1d")
    if df.empty or len(df) < WARMUP_BARS + 50:
        continue
    close_cols[sym] = df["close"]
    held_cols[sym] = spot_signal(df, SpotParams())["signal"]
cpanel = pd.concat(close_cols, axis=1).sort_index()
hpanel = pd.concat(held_cols, axis=1).reindex(cpanel.index).fillna(0.0)
ranked = ranked_spot_book(cpanel, hpanel, lookback=60, top_n=3)
equal = equal_weight_book(cpanel, hpanel)
bh = bh_equal(cpanel)
oos_idx = parts["oos"].index.intersection(ranked.index)
tbl = pd.DataFrame({
    "ranked": book_kpis(ranked.loc[oos_idx]),
    "equal": book_kpis(equal.loc[oos_idx]),
    "buyhold": book_kpis(bh.loc[oos_idx]),
}).T
display(tbl.round(3))
equity_overlay({
    "ranked top-3": ranked.loc[oos_idx]["equity"],
    "equal eligible": equal.loc[oos_idx]["equity"],
    "buy&hold equal": bh.loc[oos_idx]["equity"],
}, "OOS spot book").show()


In [ ]:
art = Path(ROOT) / "research" / "artifacts" / "lab_results.json"
if not art.exists():
    art = Path("/opt/cursor/artifacts/lab_results.json")
if art.exists():
    lab = json.loads(art.read_text())
    print("protocol", lab.get("protocol"))
    display(pd.DataFrame(lab.get("kpi_summary", {})).T.round(3) if lab.get("kpi_summary") else "no kpi_summary yet — run python -m research.trend_lab.run_lab --quick")
    for h in lab.get("hypotheses", []):
        print(f"{h.get('id')}  {h.get('result')}  — {h.get('claim')}")
else:
    print("No lab_results.json yet. From python/:  python -m research.trend_lab.run_lab --quick")


## Professional caution

* **Overfit:** Optuna on 16–28 trials of a 7-knob space will find IS luck. DF neighborhood (KAMA notebook method) is the filter; if the stable pool is empty, the fit is a spike, not a plateau.
* **Lookahead / repaint:** Donchian uses `high.shift(1).rolling`. KAMA/ALMA/EMA are causal. Fills are next-bar (`held = signal.shift(1)`). TEMA entry uses the signal bar close; SL/TP on subsequent bars; same-bar both → SL.
* **10× isolated:** a −10% adverse move wipes the stake. Headline E[R] at 10× is not a 10× Sharpe. Liquidation count is a first-class KPI.
* **Chop:** ADX < ~18 is “no trade / size 0”, not a new oscillator to fit.
* **Cross-section:** five names is a toy book. Cluster_max stops doubling ETH-beta. This is still not a 50-name futures book.
* **Live engine:** 4h A/A+ TEMA 9/90/199 is the frozen measured edge. Daily TEMA A/A+ OOS loses. This lab does not add `1d` to `SCAN_TIMEFRAMES` and does not change Pine.
